# Fine-tune InLegalBERT — Legal Contract Analyzer

Runs `pipeline/train_classifier_bert.py` from the repo on a free Colab GPU instead of the 12+ hour CPU estimate in the script's own docstring.

**Before running anything below:** `Runtime -> Change runtime type -> T4 GPU`, then `Save`.

What this notebook does:
1. Clones the repo's `claude/kind-newton-jsmo1r` branch (has the fine-tuning script + the fixed 47-class held-out evaluation).
2. Pulls `legal_contract_clauses.csv` (the CUAD training data) from the `main` branch of the same repo, since `pipeline/train_classifier.py` expects it one directory above the repo root.
3. Installs the handful of extra packages needed (Colab already ships torch + transformers).
4. Runs the fine-tune, which fine-tunes `law-ai/InLegalBERT` on the *exact* held-out split the TF-IDF+LR baseline was scored on, and writes `evaluation/results/inlegalbert_47class.json`.
5. Prints a baseline-vs-InLegalBERT comparison and downloads the results JSON.

Expect roughly 15-40 minutes on a T4 for 4 epochs over ~7,758 training rows (vs. 12+ hours on CPU).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then re-run this cell.'

Tesla T4, 15360 MiB


In [2]:
%cd /content
!rm -rf legal-contract-analyzer
!git clone -b claude/kind-newton-jsmo1r https://github.com/aniketh703/legal-contract-analyzer.git
%cd legal-contract-analyzer

/content
Cloning into 'legal-contract-analyzer'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 210 (delta 63), reused 184 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 14.64 MiB | 17.57 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/legal-contract-analyzer


In [3]:
# The CUAD training CSV lives on the `main` branch (report-assets branch), not on
# the code branch. pipeline/train_classifier.py expects it one directory above
# the repo root (CSV_PATH = ROOT.parent / "legal_contract_clauses.csv"), i.e.
# /content/legal_contract_clauses.csv when this repo is cloned to /content/legal-contract-analyzer.
!git fetch origin main --depth 1
!git show origin/main:legal_contract_clauses.csv > ../legal_contract_clauses.csv
!wc -l ../legal_contract_clauses.csv

remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/aniketh703/legal-contract-analyzer
 * branch            main       -> FETCH_HEAD
11926 ../legal_contract_clauses.csv


In [4]:
# Minimal install -- skip requirements.txt's heavier/unrelated deps
# (label-studio, faiss-cpu, pdfplumber, pymupdf, pytesseract) that this
# training script doesn't touch and that would just slow the install down.
!pip install -q -U transformers accelerate scikit-learn joblib pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 98.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2

In [5]:
# Run as a module, not as a bare script path. train_classifier_bert.py lives
# inside the pipeline/ package itself, so `python pipeline/train_classifier_bert.py`
# makes Python treat pipeline/ as the run directory, which breaks resolving
# `pipeline` as a package (it's a namespace package -- no __init__.py) once code
# inside it does `from pipeline.X import Y`. `-m` anchors sys.path to the cwd
# (the repo root) instead, which resolves it correctly.
!python -m pipeline.train_classifier_bert

[bert] 47 labels loaded from /content/legal-contract-analyzer/models/risk_map.json
[bert] Train: 7,758  |  Test: 1,940 (test set verified identical to the TF-IDF+LR baseline split)
[bert] Loading tokenizer + model: law-ai/InLegalBERT
config.json: 100% 671/671 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 100% 516/516 [00:00<00:00, 3.04MB/s]
vocab.txt: 100% 222k/222k [00:00<00:00, 21.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 482kB/s]

pytorch_model.bin: downloading bytes:  41% 220M/534M [00:01<00:01, 229MB/s, 18.8MB/s  ]
pytorch_model.bin: downloading bytes:  61% 324M/534M [00:02<00:00, 211MB/s, 27.7MB/s  ]
pytorch_model.bin: downloading bytes:  85% 454M/534M [00:02<00:00, 241MB/s, 37.7MB/s  ]
pytorch_model.bin: downloading bytes:  92% 493M/534M [00:02<00:00, 185MB/s, 41.0MB/s  ]
pytorch_model.bin: reconstructing file:  75% 402M/534M [00:03<00:01, 110MB/s, 29.9MB/s  ]
pytorch_model.bin: downloading bytes: 100% 504M/504M [00:04<00:00, 116MB/s, 42.5MB/s  ]
pytorch_model.

In [6]:
import json

with open("evaluation/results/tfidf_lr_baseline_47class.json") as f:
    baseline = json.load(f)
with open("evaluation/results/inlegalbert_47class.json") as f:
    bert = json.load(f)

b_cr = baseline["classification_report"]
n_cr = bert["classification_report"]

print(f"{'Metric':<22}{'TF-IDF + LR':>14}{'InLegalBERT':>14}")
print(f"{'Accuracy':<22}{baseline['accuracy']:>14.4f}{bert['accuracy']:>14.4f}")
print(f"{'Macro F1':<22}{b_cr['macro avg']['f1-score']:>14.4f}{n_cr['macro avg']['f1-score']:>14.4f}")
print(f"{'Weighted F1':<22}{b_cr['weighted avg']['f1-score']:>14.4f}{n_cr['weighted avg']['f1-score']:>14.4f}")

Metric                   TF-IDF + LR   InLegalBERT
Accuracy                      0.8247        0.8557
Macro F1                      0.7399        0.7080
Weighted F1                   0.8238        0.8470


In [7]:
from google.colab import files

files.download("evaluation/results/inlegalbert_47class.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: keep the fine-tuned model weights

Only run this if you want to reuse the fine-tuned model later (e.g. to wire it into the live demo) -- it's a few hundred MB, so skip it if you only need the numbers above.

In [8]:
!zip -qr inlegalbert_classifier.zip models/inlegalbert_classifier
from google.colab import files

files.download("inlegalbert_classifier.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>